In [1]:
from pathlib import Path
import json
import shutil

from pedalboard import (
    Pedalboard,
    Reverb,
    Gain,
    HighpassFilter,
    LowpassFilter,
    Compressor,
)
from pedalboard.io import AudioFile

PROJECT_ROOT = Path("/workspace/storyfx")

REVERB_PRESET_PATH = (
    PROJECT_ROOT / "configs" / "audio_presets" / "reverb_presets.json"
)

TRANSMISSION_PRESET_PATH = (
    PROJECT_ROOT / "configs" / "audio_presets" / "transmission_presets.json"
)


In [2]:
def extract_excerpt_id(path, suffix):
    path = Path(path)
    filename = path.stem

    if filename.startswith("pred_"):
        filename = filename[len("pred_"):]

    if not filename.endswith(suffix):
        raise ValueError(
            f"Expected filename ending with '{suffix}', got: {path.name}"
        )

    return filename[:-len(suffix)]


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


reverb_presets = load_json(REVERB_PRESET_PATH)
transmission_presets = load_json(TRANSMISSION_PRESET_PATH)

print(f"Loaded {len(reverb_presets)} reverb presets")
print(f"Loaded {len(transmission_presets)} transmission presets")


Loaded 12 reverb presets
Loaded 5 transmission presets


In [3]:
def read_audio(path):
    path = Path(path)

    with AudioFile(str(path)) as f:
        audio = f.read(f.frames)
        sample_rate = f.samplerate

    return audio, sample_rate


def write_audio(path, audio, sample_rate):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with AudioFile(
        str(path),
        "w",
        sample_rate,
        audio.shape[0],
    ) as f:
        f.write(audio)


In [4]:
def initialize_wet_mixes(schedule):
    schedule = Path(schedule)
    discrete_schedule = load_json(schedule)

    excerpt_id = extract_excerpt_id(
        schedule,
        "_discrete_sfx_schedule",
    )

    dry_dir = (
        PROJECT_ROOT
        / "outputs"
        / "assets"
        / excerpt_id
        / "discrete_assets"
    )

    wet_dir = (
        PROJECT_ROOT
        / "outputs"
        / "wet_mixes"
        / excerpt_id
    )

    wet_dir.mkdir(parents=True, exist_ok=True)

    copied = []

    print(f"Initializing wet mixes for {excerpt_id}")

    for sfx in discrete_schedule:
        sfx_id = sfx["discrete_sfx_id"]

        dry_path = dry_dir / f"{sfx_id}.wav"
        wet_path = wet_dir / f"{sfx_id}.wav"

        if not dry_path.exists():
            raise FileNotFoundError(
                f"Dry asset not found: {dry_path}"
            )

        shutil.copy2(dry_path, wet_path)
        copied.append(wet_path)

        print(f"[COPY] {sfx_id}")

    print(f"\nInitialized {len(copied)} wet mixes.")

    return wet_dir


In [10]:
def build_transmission_board(preset):
    effects = []

    highpass_hz = preset.get("highpass_hz")
    lowpass_hz = preset.get("lowpass_hz")
    compression = preset.get("compression")
    gain_db = preset.get("gain_db")

    if highpass_hz is not None:
        effects.append(
            HighpassFilter(
                cutoff_frequency_hz=highpass_hz
            )
        )

    if lowpass_hz is not None:
        effects.append(
            LowpassFilter(
                cutoff_frequency_hz=lowpass_hz
            )
        )

    if compression is not None:
        effects.append(
            Compressor(
                threshold_db=compression["threshold_db"],
                ratio=compression["ratio"],
                attack_ms=compression["attack_ms"],
                release_ms=compression["release_ms"],
            )
        )

    if gain_db is not None:
        effects.append(
            Gain(gain_db=gain_db)
        )

    return Pedalboard(effects)


In [5]:
def transmission_profile_activation(schedule):
    schedule = Path(schedule)
    discrete_schedule = load_json(schedule)

    excerpt_id = extract_excerpt_id(
        schedule,
        "_discrete_sfx_schedule",
    )

    wet_dir = (
        PROJECT_ROOT
        / "outputs"
        / "wet_mixes"
        / excerpt_id
    )

    if not wet_dir.exists():
        raise FileNotFoundError(
            "Wet-mix directory does not exist. "
            "Run initialize_wet_mixes(schedule) first."
        )

    print(
        f"Applying transmission profiles for {excerpt_id}"
    )

    processed_count = 0

    for sfx in discrete_schedule:
        sfx_id = sfx["discrete_sfx_id"]
        profile_name = sfx.get("transmission_profile")
        wet_path = wet_dir / f"{sfx_id}.wav"

        if not wet_path.exists():
            raise FileNotFoundError(
                f"Wet mix not found: {wet_path}"
            )

        if profile_name is None:
            print(
                f"[TRANSMISSION] {sfx_id}: none"
            )
            continue

        if profile_name not in transmission_presets:
            raise KeyError(
                f"Unknown transmission profile "
                f"'{profile_name}' for {sfx_id}"
            )

        preset = transmission_presets[profile_name]
        board = build_transmission_board(preset)

        audio, sample_rate = read_audio(wet_path)
        processed = board(audio, sample_rate)

        write_audio(
            wet_path,
            processed,
            sample_rate,
        )

        processed_count += 1

        print(
            f"[TRANSMISSION] {sfx_id}: {profile_name}"
        )

    print(
        f"\nTransmission processing complete: "
        f"{processed_count} asset(s) processed."
    )


In [6]:
def build_reverb_board(preset):
    effects = []

    eq = preset.get("eq")

    if eq is not None:
        highpass_hz = eq.get("highpass_hz")
        lowpass_hz = eq.get("lowpass_hz")

        if highpass_hz is not None:
            effects.append(
                HighpassFilter(
                    cutoff_frequency_hz=highpass_hz
                )
            )

        if lowpass_hz is not None:
            effects.append(
                LowpassFilter(
                    cutoff_frequency_hz=lowpass_hz
                )
            )

    reverb = preset.get("reverb")

    if reverb is not None:
        effects.append(
            Reverb(
                room_size=reverb["room_size"],
                damping=reverb["damping"],
                wet_level=reverb["wet_level"],
                dry_level=reverb["dry_level"],
                width=reverb["width"],
                freeze_mode=reverb.get(
                    "freeze_mode",
                    0.0,
                ),
            )
        )

    return Pedalboard(effects)


In [7]:
def reverb_profile_activation(schedule):
    schedule = Path(schedule)
    discrete_schedule = load_json(schedule)

    excerpt_id = extract_excerpt_id(
        schedule,
        "_discrete_sfx_schedule",
    )

    wet_dir = (
        PROJECT_ROOT
        / "outputs"
        / "wet_mixes"
        / excerpt_id
    )

    if not wet_dir.exists():
        raise FileNotFoundError(
            "Wet-mix directory does not exist. "
            "Run initialize_wet_mixes(schedule) first."
        )

    print(
        f"Applying reverb profiles for {excerpt_id}"
    )

    processed_count = 0

    for sfx in discrete_schedule:
        sfx_id = sfx["discrete_sfx_id"]
        profile_name = sfx.get("reverb_profile")
        wet_path = wet_dir / f"{sfx_id}.wav"

        if not wet_path.exists():
            raise FileNotFoundError(
                f"Wet mix not found: {wet_path}"
            )

        if profile_name is None:
            print(
                f"[REVERB] {sfx_id}: none"
            )
            continue

        if profile_name not in reverb_presets:
            raise KeyError(
                f"Unknown reverb profile "
                f"'{profile_name}' for {sfx_id}"
            )

        preset = reverb_presets[profile_name]
        board = build_reverb_board(preset)

        audio, sample_rate = read_audio(wet_path)
        processed = board(audio, sample_rate)

        write_audio(
            wet_path,
            processed,
            sample_rate,
        )

        processed_count += 1

        eq_note = (
            " + EQ"
            if preset.get("eq") is not None
            else ""
        )

        print(
            f"[REVERB] {sfx_id}: "
            f"{profile_name}{eq_note}"
        )

    print(
        f"\nReverb processing complete: "
        f"{processed_count} asset(s) processed."
    )


In [8]:
DISCRETE_SCHEDULE = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / "excerpt_001_discrete_sfx_schedule.json"
)

initialize_wet_mixes(
    DISCRETE_SCHEDULE
)

transmission_profile_activation(
    DISCRETE_SCHEDULE
)

reverb_profile_activation(
    DISCRETE_SCHEDULE
)


Initializing wet mixes for excerpt_001
[COPY] discrete_sfx000
[COPY] discrete_sfx001
[COPY] discrete_sfx002
[COPY] discrete_sfx003
[COPY] discrete_sfx004
[COPY] discrete_sfx005
[COPY] discrete_sfx006
[COPY] discrete_sfx007
[COPY] discrete_sfx008
[COPY] discrete_sfx009
[COPY] discrete_sfx010
[COPY] discrete_sfx011
[COPY] discrete_sfx012
[COPY] discrete_sfx013
[COPY] discrete_sfx014
[COPY] discrete_sfx015
[COPY] discrete_sfx016

Initialized 17 wet mixes.
Applying transmission profiles for excerpt_001
[TRANSMISSION] discrete_sfx000: none
[TRANSMISSION] discrete_sfx001: none
[TRANSMISSION] discrete_sfx002: none
[TRANSMISSION] discrete_sfx003: none
[TRANSMISSION] discrete_sfx004: none
[TRANSMISSION] discrete_sfx005: none
[TRANSMISSION] discrete_sfx006: none
[TRANSMISSION] discrete_sfx007: none
[TRANSMISSION] discrete_sfx008: none
[TRANSMISSION] discrete_sfx009: none
[TRANSMISSION] discrete_sfx010: none
[TRANSMISSION] discrete_sfx011: none
[TRANSMISSION] discrete_sfx012: none
[TRANSMISSION] 